# NIH14 CBM Evaluation Notebook

This notebook mirrors `evaluate_cbm.ipynb` but targets the NIH14 multilabel setting. It loads a trained CBM, computes logits on the validation split, sweeps per-class thresholds (optional), prints metrics, and visualizes the confusion matrix just like `evaluate_cbm_nih.py`.

In [ ]:
import os
import json
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

os.chdir('..')

import cbm
import data_utils
import evaluate_cbm_nih as eval_nih


In [ ]:
MODEL_DIR = 'saved_models/nih14_example'
NIH_IMG_DIR = os.environ.get('NIH_CXR_IMG_DIR', '/path/to/NIH/images-224')
SPLIT = 'nih14_val'
DEVICE = 'cuda'
BATCH_SIZE = 128
SWEEP_THRESHOLDS = True
SWEEP_STEPS = 201
SWEEP_METRIC = 'f1'  # or 'precision'
THRESHOLDS_PATH = None  # path to JSON if you want to reuse thresholds
CONFUSION_PLOT_PATH = None


In [ ]:
with open(os.path.join(MODEL_DIR, 'args.txt')) as f:
    model_args = json.load(f)
views = model_args.get('nih_views', 'PA')
view_list = [v.strip() for v in views.split(',') if v.strip()]
nih_cfg = {
    'img_dir': NIH_IMG_DIR,
    'csv_path': model_args.get('nih_csv_path'),
    'train_fraction': model_args.get('nih_train_fraction', 0.9),
    'split_seed': model_args.get('nih_split_seed', 0),
    'views': view_list,
    'batch_size': BATCH_SIZE,
}
backbone = eval_nih.load_backbone_name(MODEL_DIR)
loader, dataset = eval_nih.prepare_dataset(SPLIT, backbone, nih_cfg, num_workers=4)
model = cbm.load_cbm(MODEL_DIR, DEVICE)
model.eval()
print(f'Loaded CBM from {MODEL_DIR} with backbone {backbone}. Dataset size: {len(dataset)} samples.')


In [ ]:
all_probs, all_targets = [], []
with torch.no_grad():
    for images, labels in tqdm(loader, desc='Computing logits'):
        images = images.to(DEVICE)
        logits, _ = model(images)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
        all_targets.append(labels.float())

probs = torch.cat(all_probs, dim=0).numpy()
targets = torch.cat(all_targets, dim=0).numpy()
print('Probabilities shape:', probs.shape)


In [ ]:
classes_path = data_utils.LABEL_FILES['nih14']
with open(classes_path) as f:
    classes = [c for c in f.read().splitlines() if c]
threshold_values = None
if SWEEP_THRESHOLDS:
    thr, scores, accs, precs, recs, label = eval_nih.sweep_thresholds(probs, targets, SWEEP_STEPS, metric=SWEEP_METRIC)
    threshold_values = np.where(np.isnan(thr), 0.5, thr)
    sweep_info = (thr, scores, accs, precs, recs, label)
elif THRESHOLDS_PATH:
    threshold_values = eval_nih._load_thresholds(THRESHOLDS_PATH, classes)
metrics = eval_nih.compute_metrics(probs, targets, threshold=0.5, classes=classes, per_class_thresholds=threshold_values)
threshold_label = 'per-class' if threshold_values is not None else '0.5'
print('Exact match:', metrics['exact_match'])
print('Micro precision:', metrics['micro_precision'])
print('Micro recall:', metrics['micro_recall'])
print('Mean AUROC:', metrics['mean_auroc'])
print('Mean AP:', metrics['mean_ap'])


In [ ]:
import pandas as pd
rows = []
conf = metrics['confusion']
for idx, cls in enumerate(metrics['classes']):
    rows.append({
        'class': cls,
        'auroc': metrics['aurocs'][idx],
        'ap': metrics['aps'][idx],
        'accuracy': metrics['per_class_accuracy'][idx],
        'tp': conf['tp'][idx],
        'tn': conf['tn'][idx],
        'fp': conf['fp'][idx],
        'fn': conf['fn'][idx],
    })
pd.DataFrame(rows)


In [ ]:
if SWEEP_THRESHOLDS:
    thr, scores, accs, precs, recs, label = sweep_info
    for cls, t, s, p, r in zip(classes, thr, scores, precs, recs):
        print(f'{cls:20s} thr={t:.3f} {label}={s:.4f} prec={p:.3f} rec={r:.3f}')


In [ ]:
eval_nih.plot_confusion(classes, metrics['confusion'], threshold_label, path=CONFUSION_PLOT_PATH)
